In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-0"

In [ ]:
# helper functions that will help maintain the conversation state with claude
def add_user_message(messages, text):
    user_message = {"role":"user","content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role":"assistant","content":text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params={
        "model":model,
        "max_tokens":1000,
        "messages":messages,
        "temperature":temperature
    }
    if system:
        params["system"]=system
    if stop_sequences:
        params["stop_sequences"]=stop_sequences
        
    message=client.messages.create(**params)
    
    return message.content[0].text


In [ ]:
messages=[]

In [ ]:
for i in range(2):
    print("Enter the question you want to ask")
    user_request = input("Enter the question you want claude to answer: ")
    add_user_message(messages,user_request)
    #without system prompts we have to do this
    assistant_text=chat(messages)
    add_assistant_message(messages,assistant_text)
    print("---")
    print(f"{i+1} text: ",assistant_text)
    print("---")

In [ ]:
# UNDERSTANDING SYSTEM PROMPTS IN CLAUDE
system_prompt="""You are a Python Enginner who writes very clean and concise code.
"""
messages.clear()
add_user_message(messages,"Write a python function to add two numbers")
print("messages dictionary:",messages)
#with system prompts we do this
# answer = chat(messages, system=system_prompt)
# print(answer)

In [ ]:
messages.clear()
add_user_message(messages,"write a 1 sentence on moon")
with client.messages.stream(model=model,max_tokens=1000,messages=messages) as stream:
    for text in stream.text_stream:
        print(text, end="")

In [ ]:
#STRUCTURED DATA
messages.clear()
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages,"```json")
text = chat(messages, stop_sequences=["```"])
print(text)

In [ ]:
#strcutured data exercise
messages.clear()
prompt = """Generate three different AWS CLI commands. Each should be very short"""
add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all the bash commands in a single block without any comments:\n```bash")
# here we add a pre filled message to claude to make generation look good and as we want it
#stop sequences make sure that generation stops when those symbols are encountered.
text=chat(messages,stop_sequences=["```","**"])
print(text)

In [ ]:
# prompt evaluation starts here
def generate_dataset():
    prompt=""" Generate an evaluation dataset for a prompt evaluation. The dataset will be used to 
     evaluate prompts that generate python, json or regex specifically for AWS related tasks.
      Generate an array of JSON objects, each representing task that requires Python,json or a regex to complete 
    
     Example output:
     ```json
     [
     {
     "task":"Description of task",
     "format":"json" or "python" or "regex"
     },
     ...additional
     ]
     ```

     *Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
     *Focus on tasks that do not require writing much code

     Please generate 3 objects. 
    """
    messages.clear()
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

dataset = generate_dataset()
print(dataset)

In [ ]:
# opening a file and saving the dataset in it without "with" we will have to close it manually
with open("dataset.json","w") as f:
    json.dump(dataset,f,indent=2)